# LigandMPNN Inverse Folding

![LigandMPNN Inverse Folding](https://proto-bio.github.io/proto-assets/images/tool/ligandmpnn/hero.png)

This notebook demonstrates how to use `run_ligandmpnn_sample` to design protein sequences conditioned on both backbone structure and nearby ligand atoms, and `run_ligandmpnn_score` to score sequence-structure compatibility in the same context. LigandMPNN extends ProteinMPNN by incorporating small molecule geometry into sequence design, enabling the creation of binding pocket sequences that are compatible with a specific ligand conformation. When no ligand atoms are present in the input structure, LigandMPNN behaves similarly to ProteinMPNN.


In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ligandmpnn")
display_overview("ligandmpnn")
display_docs_section("ligandmpnn", "Background")

## Available tools

In [ ]:
display_available_tools("ligandmpnn")

### `run_ligandmpnn_sample`

LigandMPNN analyzes the backbone coordinates and any ligand atoms present in the structure to generate sequences optimized for the target fold and binding environment. We use GFP as a minimal runnable example. For ligand-aware design, provide a structure file that includes HETATM records for the bound ligand of interest. You can optionally fix residue positions to preserve critical interactions and exclude specific amino acids from the designed sequences.

In [ ]:
from proto_tools import (
    ESMFoldInput,
    InverseFoldingStructureInput,
    LigandMPNNSampleConfig,
    LigandMPNNSampleInput,
    LigandMPNNScoringConfig,
    LigandMPNNScoringInput,
    SequenceStructurePair,
    Complex,
    run_esmfold,
    run_ligandmpnn_sample,
    run_ligandmpnn_score,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [ ]:
# Display input docs
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_sample")

# Load GFP structure and configure design constraints
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    fixed_positions={"A": [2, 3, 4]},
)
inputs = LigandMPNNSampleInput(inputs=[struct_input])

In [ ]:
# Display config docs
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_sample")

# Configure sampling | see docs above for all fields
config = LigandMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs
    excluded_amino_acids=["C"],     # Exclude cysteine
    seed=42,
    device="cuda",                  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling function
result = run_ligandmpnn_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_sample")

# Each input structure yields a LigandMPNNDesignSet of LigandMPNNDesign complexes.
# A LigandMPNNDesign holds every chain of the input (designed + fixed context)
# plus per-design recovery metrics.
import math

for i, design_set in enumerate(result.design_sets):
    print(f"Structure {i}: {len(design_set.complexes)} complexes")
    for j, design in enumerate(design_set.complexes, 1):
        recovery = design.metrics["sequence_recovery"]
        interface = design.metrics.get("ligand_interface_sequence_recovery", float("nan"))
        iface_str = "N/A (no ligand interface)" if math.isnan(interface) else f"{interface:.3f}"
        for chain, was_designed in zip(design.chains, design.designed, strict=True):
            tag = "designed" if was_designed else "fixed context"
            preview = f"{chain.sequence[:50]}..." if len(chain.sequence) > 50 else chain.sequence
            print(f"  Design {j} chain {chain.id} ({tag}, {len(chain.sequence)} aa): {preview}")
        print(f"  Design {j} metrics: recovery={recovery:.3f} | interface={iface_str}")

### Fold-validating a design

A `LigandMPNNDesign` already carries every chain of the complex, so it can be handed straight to a structure predictor as `design` to check that the designed sequence folds back to the target without manually reassembling chains.

In [ ]:
# A LigandMPNNDesign drops straight into a structure predictor.
design = result.design_sets[0].complexes[0]
fold = run_esmfold(ESMFoldInput(complexes=[design]))
print(f"Folded {len(design.chains)} chains | success={fold.success}")

### `run_ligandmpnn_score`

Use `run_ligandmpnn_score` to compute LigandMPNN log-likelihood and perplexity for a sequence in a structure context. This example loads a small protein-only backbone (the in-tree `example_input_fixture.pdb`) and scores its native chain-A sequence — LigandMPNN's score path requires a fully-canonical polymer chain, so chromophore / non-standard residues stay on the sampling side demonstrated above.


In [ ]:
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_score")

# Resolve the in-tree clean-protein fixture by walking up to the repo root.
from pathlib import Path
from proto_tools.entities.structures import Structure
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
score_structure = Structure.from_file(str(_repo_root / "proto_tools" / "tools" / "inverse_folding" / "example_input_fixture.pdb"))
score_sequence = score_structure.get_chain_sequence("A", remove_non_standard=True)

score_inputs = LigandMPNNScoringInput(
    sequence_structure_pairs=[SequenceStructurePair(sequence=score_sequence, structure=score_structure)]
)
score_result = run_ligandmpnn_score(
    score_inputs,
    LigandMPNNScoringConfig(seed=42, return_logits=False),
)

score = score_result.scores[0]
print(f"Perplexity: {score.perplexity:.3f}")
print(f"Average log-likelihood: {score.avg_log_likelihood:.3f}")

## Export Results

Designed sequences can be saved to standard file formats for downstream analysis. JSON format is convenient for preserving the full LigandMPNN metrics alongside the sequences, while FASTA format is useful for feeding results into other sequence analysis tools.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="ligandmpnn_designs", export_path=output_dir, file_format="json")
score_result.export(name="ligandmpnn_scores", export_path=output_dir, file_format="json")
print(f"Results exported to {output_dir}")